In [2]:
import torch

In [3]:
# consider a function in 3d
def y_fun(x):
    return (x[:,0]**4 - (x[:,1]**3 + x[:,2]**4) + x[:,1]*x[:,2]).unsqueeze(dim=1)
    #return (x[:,0]**3 - (x[:,1]**2 + x[:,2]**2))

x = torch.tensor([
    [0.0, 1.0, 2.0],
    [1.0, 2.0, 0.0],
    [2.0, 0.0, 1.0],
    [1.0, 1.0, 1.0],
    [-1.0, -1.0, -1.0],
], requires_grad=True)
bs, d = x.shape
bs, d

(5, 3)

In [4]:
y = y_fun(x)
y

tensor([[-15.],
        [ -7.],
        [ 15.],
        [  0.],
        [  2.]], grad_fn=<UnsqueezeBackward0>)

In [5]:
g = torch.autograd.grad(
    inputs=x,
    outputs=y,
    grad_outputs=torch.ones_like(y),
    create_graph=True,
    retain_graph=True,
)[0]
g

tensor([[  0.,  -1., -31.],
        [  4., -12.,   2.],
        [ 32.,   1.,  -4.],
        [  4.,  -2.,  -3.],
        [ -4.,  -4.,   3.]], grad_fn=<AddBackward0>)

In [6]:
l0 = torch.autograd.grad(
    inputs=x,
    outputs=g[:,0],
    grad_outputs=torch.ones_like(g[:,0]),
    retain_graph=True,
    create_graph=False,
)[0]
l0

tensor([[ 0.,  0.,  0.],
        [12.,  0.,  0.],
        [48.,  0.,  0.],
        [12.,  0.,  0.],
        [12.,  0.,  0.]])

In [7]:
l1 = torch.autograd.grad(
    inputs=x,
    outputs=g[:,1],
    grad_outputs=torch.ones_like(g[:,1]),
    retain_graph=True,
    create_graph=False,
)[0]
l1

tensor([[  0.,  -6.,   1.],
        [  0., -12.,   1.],
        [  0.,   0.,   1.],
        [  0.,  -6.,   1.],
        [  0.,   6.,   1.]])

In [8]:
l2 = torch.autograd.grad(
    inputs=x,
    outputs=g[:,2],
    grad_outputs=torch.ones_like(g[:,2]),
    retain_graph=False,
    create_graph=False,
)[0]
l2

tensor([[  0.,   1., -48.],
        [  0.,   1.,   0.],
        [  0.,   1., -12.],
        [  0.,   1., -12.],
        [  0.,   1., -12.]])

In [9]:
d2u_dx2 = l0[:,0]
d2u_dy2 = l1[:,1]
d2u_dz2 = l2[:,2]
d2u_dx2.sum(), d2u_dy2.sum(), d2u_dz2.sum()

(tensor(84.), tensor(-18.), tensor(-84.))

---

In [35]:
g = torch.autograd.grad(
    inputs=x,
    outputs=y,
    grad_outputs=torch.ones_like(y),
    create_graph=True,
    retain_graph=True,
)[0]
g

tensor([[  0.,  -1., -31.],
        [  4., -12.,   2.],
        [ 32.,   1.,  -4.],
        [  4.,  -2.,  -3.],
        [ -4.,  -4.,   3.]], grad_fn=<AddBackward0>)

In [36]:
laplace = 0
for i in range(d):
    hess_row = torch.autograd.grad(
        inputs=x,
        outputs=g[:,i].sum(),
        grad_outputs=torch.tensor(1.0),
        retain_graph=i<d-1,
        create_graph=False,
    )[0]
    second_ders_bs = hess_row[:,i]
    print(second_ders_bs)
laplace

tensor([ 0., 12., 48., 12., 12.])
tensor([ -6., -12.,   0.,  -6.,   6.])
tensor([-48.,   0., -12., -12., -12.])


0